In [1]:
import json

import numpy as np
import plotly.graph_objects as go
from scipy.linalg import subspace_angles
from sklearn.decomposition import PCA

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader
from miscope.analysis.library import extract_neuron_weight_matrix

In [2]:
FAMILY_NAME = "modulo_addition_1layer"
family = load_family(FAMILY_NAME)

PRIME = 113

# k — global W_in subspace dimensionality for cross-variant comparison.
# From macro surface analysis: ~10% variance per component → ~12 effective dims.
# Start at 6 (conservative) and 12 (full expected dimensionality).
N_GLOBAL_COMPONENTS = 6

# k per frequency group — within-group PCA is 3D
N_GROUP_COMPONENTS = 3

_HEALTH_COLORS = {
    "healthy": "#2ca02c",
    "degraded": "#d62728",
    "unknown": "#7f7f7f",
}

## Setup

**Question:** Do models within the same prime group traverse the same low-dimensional manifold?

If different seeds converge to the same W_in subspace, the manifold is an attractor determined
by the task structure (mod p arithmetic).  If seeds diverge, the manifold is initialization-sensitive.

**Metric:** Principal angles between k-dimensional subspaces (Grassmannian distance).
- 0° = subspaces are identical
- 90° = subspaces are orthogonal
- `scipy.linalg.subspace_angles(A, B)` takes column-span matrices → `basis.T` shape `(d_model, k)`

Two levels:
1. **Global** — PCA over all d_mlp neurons, k=6 components
2. **Per-group** — PCA within each frequency group's neurons, k=3 components; compare same freq across variants

In [3]:
def get_health_label(variant) -> str:
    summary_path = variant.variant_dir / "variant_summary.json"
    if not summary_path.exists():
        return "unknown"
    summary = json.loads(summary_path.read_text())
    clf = summary.get("performance_classification", {})
    label = clf.get("label", "unknown") if isinstance(clf, dict) else str(clf)
    return "healthy" if "healthy" in label.lower() else "degraded"


def extract_win_subspace(variant, n_components: int) -> np.ndarray:
    """Fit PCA on final-epoch W_in, all neurons. Returns (n_components, d_model)."""
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    epochs = loader.get_epochs("parameter_snapshot")
    snap = loader.load_epoch("parameter_snapshot", epochs[-1])
    W_in = extract_neuron_weight_matrix(snap)  # (d_model, d_mlp)
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(W_in.T)  # neurons as samples: (d_mlp, d_model)
    return pca.components_  # (n_components, d_model)


def extract_group_subspaces(variant, n_components: int = 3) -> dict[int, np.ndarray]:
    """Per-frequency-group W_in subspace at final epoch.

    Returns:
        {freq_1indexed: (n_components, d_model)} for each group with enough neurons.
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    epochs = loader.get_epochs("parameter_snapshot")
    snap = loader.load_epoch("parameter_snapshot", epochs[-1])
    W_in = extract_neuron_weight_matrix(snap)  # (d_model, d_mlp)

    ngpca = loader.load_cross_epoch("neuron_group_pca")
    group_idx = ngpca["neuron_group_idx"]                    # (d_mlp,)
    group_freqs = (ngpca["group_freqs"] + 1).astype(int)     # 1-indexed

    subspaces = {}
    for g, freq in enumerate(group_freqs):
        mask = group_idx == g
        if mask.sum() < n_components:
            continue
        neurons = W_in.T[mask]  # (n_neurons, d_model)
        pca = PCA(n_components=n_components, random_state=0)
        pca.fit(neurons)
        subspaces[int(freq)] = pca.components_  # (n_components, d_model)
    return subspaces


def pairwise_principal_angles(bases: list[np.ndarray]) -> np.ndarray:
    """Mean principal angle (degrees) for all pairs. bases[i] shape (k, d_model)."""
    n = len(bases)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            angles = subspace_angles(bases[i].T, bases[j].T)  # (d_model, k) each
            val = float(np.degrees(angles.mean()))
            matrix[i, j] = val
            matrix[j, i] = val
    return matrix

## Load all p=113 variants

In [4]:
all_variants = family.list_variants()
p_variants = [v for v in all_variants if v.params.get("prime") == PRIME]
print(f"Found {len(p_variants)} p={PRIME} variants")

records = []
for v in p_variants:
    seed = v.params["seed"]
    data_seed = v.params["data_seed"]
    health = get_health_label(v)
    records.append({
        "variant": v,
        "label": f"s{seed}/ds{data_seed}",
        "health": health,
    })

# Sort: healthy first, then alphabetically within each class
records.sort(key=lambda r: (0 if r["health"] == "healthy" else 1, r["label"]))

for r in records:
    print(f"  {r['label']:20s}  {r['health']}")

Found 6 p=113 variants
  s485/ds598            healthy
  s999/ds598            healthy
  s485/ds42             degraded
  s485/ds999            degraded
  s999/ds42             degraded
  s999/ds999            degraded


In [5]:
print(f"Extracting global W_in subspace ({N_GLOBAL_COMPONENTS}-D) for each variant...")
for r in records:
    r["global_basis"] = extract_win_subspace(r["variant"], N_GLOBAL_COMPONENTS)
    print(f"  {r['label']:20s}  shape: {r['global_basis'].shape}")

print("\nExtracting per-group subspaces...")
for r in records:
    r["group_subspaces"] = extract_group_subspaces(r["variant"], N_GROUP_COMPONENTS)
    freqs = sorted(r["group_subspaces"].keys())
    print(f"  {r['label']:20s}  freqs: {freqs}")

Extracting global W_in subspace (6-D) for each variant...
  s485/ds598            shape: (6, 128)
  s999/ds598            shape: (6, 128)
  s485/ds42             shape: (6, 128)
  s485/ds999            shape: (6, 128)
  s999/ds42             shape: (6, 128)
  s999/ds999            shape: (6, 128)

Extracting per-group subspaces...
  s485/ds598            freqs: [27, 36, 53]
  s999/ds598            freqs: [9, 33, 38, 55]
  s485/ds42             freqs: [6, 19, 36, 49]
  s485/ds999            freqs: [13, 15, 36, 40]
  s999/ds42             freqs: [19, 20, 46, 50]
  s999/ds999            freqs: [3, 13, 25, 33]


## Global subspace alignment

Pairwise mean principal angle between each variant's global k-D W_in subspace.
Low angle = variants converged to the same manifold.  High angle = diverged.

In [6]:
n = len(records)
global_angle_matrix = pairwise_principal_angles([r["global_basis"] for r in records])

health_labels = [
    f"{r['label']} ({'H' if r['health']=='healthy' else 'D'})"
    for r in records
]

fig = go.Figure(go.Heatmap(
    z=global_angle_matrix,
    x=health_labels,
    y=health_labels,
    colorscale="RdBu_r",
    zmid=45, zmin=0, zmax=90,
    text=np.round(global_angle_matrix, 1),
    texttemplate="%{text}°",
    colorbar=dict(title="Mean principal<br>angle (°)"),
))
fig.update_layout(
    title=f"p={PRIME} global W_in subspace alignment — {N_GLOBAL_COMPONENTS}-D basis",
    width=700, height=650,
    xaxis=dict(tickangle=-45),
    margin=dict(l=120, b=120),
)
fig.show()

In [7]:
# Summary: within-class vs. across-class mean angles
healthy_idx  = [i for i, r in enumerate(records) if r["health"] == "healthy"]
degraded_idx = [i for i, r in enumerate(records) if r["health"] == "degraded"]

def _group_mean(idx_a, idx_b, mat):
    vals = [mat[i, j] for i in idx_a for j in idx_b if i != j]
    return float(np.mean(vals)) if vals else float("nan")

print(f"Global {N_GLOBAL_COMPONENTS}-D subspace (mean principal angle):")
print(f"  Healthy  ↔ Healthy:   {_group_mean(healthy_idx,  healthy_idx,  global_angle_matrix):.1f}°")
print(f"  Degraded ↔ Degraded:  {_group_mean(degraded_idx, degraded_idx, global_angle_matrix):.1f}°")
print(f"  Healthy  ↔ Degraded:  {_group_mean(healthy_idx,  degraded_idx, global_angle_matrix):.1f}°")

Global 6-D subspace (mean principal angle):
  Healthy  ↔ Healthy:   80.1°
  Degraded ↔ Degraded:  77.9°
  Healthy  ↔ Degraded:  76.9°


## Per-group subspace alignment

For each frequency shared across all variants, compare the 3-D subspace occupied by
that group's neurons.  Same-frequency neurons doing the same job — do they land in the
same weight-space region regardless of seed?

In [8]:
all_freq_sets = [set(r["group_subspaces"].keys()) for r in records]
common_freqs = sorted(set.intersection(*all_freq_sets))
print(f"Frequencies present in all {len(records)} variants: {common_freqs}")

per_group_matrices = {}
for freq in common_freqs:
    bases = [r["group_subspaces"][freq] for r in records]
    per_group_matrices[freq] = pairwise_principal_angles(bases)

for freq, mat in per_group_matrices.items():
    fig = go.Figure(go.Heatmap(
        z=mat,
        x=health_labels,
        y=health_labels,
        colorscale="RdBu_r",
        zmid=45, zmin=0, zmax=90,
        text=np.round(mat, 1),
        texttemplate="%{text}°",
        colorbar=dict(title="Mean principal<br>angle (°)"),
    ))
    fig.update_layout(
        title=f"p={PRIME} freq {freq} — per-group subspace alignment ({N_GROUP_COMPONENTS}-D)",
        width=650, height=580,
        xaxis=dict(tickangle=-45),
        margin=dict(l=120, b=120),
    )
    fig.show()

Frequencies present in all 6 variants: []


In [9]:
# Aggregate per-group summary
print(f"Per-group {N_GROUP_COMPONENTS}-D subspace (mean principal angle):")
print(f"{'Freq':>6}  {'H↔H':>8}  {'D↔D':>8}  {'H↔D':>8}")
for freq, mat in per_group_matrices.items():
    hh = _group_mean(healthy_idx,  healthy_idx,  mat)
    dd = _group_mean(degraded_idx, degraded_idx, mat)
    hd = _group_mean(healthy_idx,  degraded_idx, mat)
    print(f"{freq:>6}  {hh:>7.1f}°  {dd:>7.1f}°  {hd:>7.1f}°")

Per-group 3-D subspace (mean principal angle):
  Freq       H↔H       D↔D       H↔D


## Sensitivity to k

Mean principal angle across k=2..12.  Checks whether the alignment signal is
stable across subspace dimensionality choices or only visible at a specific k.

In [10]:
k_range = list(range(2, 13))
hh_by_k, dd_by_k, hd_by_k = [], [], []

for k in k_range:
    bases_k = [extract_win_subspace(r["variant"], k) for r in records]
    mat_k = pairwise_principal_angles(bases_k)
    hh_by_k.append(_group_mean(healthy_idx,  healthy_idx,  mat_k))
    dd_by_k.append(_group_mean(degraded_idx, degraded_idx, mat_k))
    hd_by_k.append(_group_mean(healthy_idx,  degraded_idx, mat_k))

fig = go.Figure()
for vals, name, color in [
    (hh_by_k, "Healthy ↔ Healthy",   _HEALTH_COLORS["healthy"]),
    (dd_by_k, "Degraded ↔ Degraded", _HEALTH_COLORS["degraded"]),
    (hd_by_k, "Healthy ↔ Degraded",  "#7f7f7f"),
]:
    fig.add_trace(go.Scatter(
        x=k_range, y=vals, mode="lines+markers",
        name=name, line=dict(color=color),
    ))
fig.update_layout(
    title=f"p={PRIME} — global subspace alignment vs. subspace dimension k",
    xaxis_title="k (subspace dimensions)",
    yaxis_title="Mean principal angle (°)",
    yaxis_range=[0, 90],
    width=700, height=420,
)
fig.show()

## Within-seed alignment

Cross-prime analysis revealed that **model seed determines anchor frequency** — seed 485 reliably
produces freq 36 across data seeds; seed 999 produces freq 33.  Data seeds modulate which
secondary frequencies appear, but the anchor is seed-level.

This section compares variants within the same model seed.  The anchor frequency is the one
present across all data seeds for that seed — the true seed-level attractor.  The question:
do same-seed variants occupy the *same* 3-D subspace for their anchor frequency?

In [11]:
from collections import defaultdict

# Group records by model seed
by_seed = defaultdict(list)
for r in records:
    by_seed[r["variant"].params["seed"]].append(r)

for seed, seed_records in sorted(by_seed.items()):
    freq_sets = [set(r["group_subspaces"].keys()) for r in seed_records]
    anchor_freqs = sorted(set.intersection(*freq_sets))
    print(f"seed={seed}: variants={[r['label'] for r in seed_records]}")
    print(f"  anchor freqs (shared across all data seeds): {anchor_freqs}")

    for freq in anchor_freqs:
        bases = [r["group_subspaces"][freq] for r in seed_records]
        mat = pairwise_principal_angles(bases)
        seed_health = ["H" if r["health"] == "healthy" else "D" for r in seed_records]
        annotated = [f"{r['label']} ({h})" for r, h in zip(seed_records, seed_health)]

        fig = go.Figure(go.Heatmap(
            z=mat,
            x=annotated, y=annotated,
            colorscale="RdBu_r",
            zmid=45, zmin=0, zmax=90,
            text=np.round(mat, 1),
            texttemplate="%{text}°",
            colorbar=dict(title="Mean principal<br>angle (°)"),
        ))
        fig.update_layout(
            title=f"p={PRIME} seed={seed} — freq {freq} subspace alignment (within-seed)",
            width=520, height=460,
            xaxis=dict(tickangle=-45),
            margin=dict(l=130, b=130),
        )
        fig.show()
    print()

seed=485: variants=['s485/ds598', 's485/ds42', 's485/ds999']
  anchor freqs (shared across all data seeds): [36]



seed=999: variants=['s999/ds598', 's999/ds42', 's999/ds999']
  anchor freqs (shared across all data seeds): []



## Phase diagram preview — does the basin show at epoch 0?

The attractor basin hypothesis: model seed selects a frequency basin *at initialization*,
before any gradient signal.  If true, the epoch-0 W_in subspace should already show
seed-level clustering — same-seed variants closer to each other, different-seed variants
further apart.

Compare pairwise subspace angles at **epoch 0** vs **final epoch**:
- If same-seed angles are already smaller at epoch 0 → basin is encoded in initialization; training sharpens it
- If epoch-0 angles are uniform across all pairs → the gradient signal (shaped by the data) does the selection

In [12]:
def extract_win_subspace_at_epoch(variant, epoch: int, n_components: int) -> np.ndarray:
    """W_in subspace at a specific epoch. Returns (n_components, d_model)."""
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    snap = loader.load_epoch("parameter_snapshot", epoch)
    W_in = extract_neuron_weight_matrix(snap)
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(W_in.T)
    return pca.components_


print("Extracting epoch-0 W_in subspaces...")
init_bases = []
for r in records:
    loader = ArtifactLoader(str(r["variant"].variant_dir / "artifacts"))
    first_epoch = loader.get_epochs("parameter_snapshot")[0]
    basis = extract_win_subspace_at_epoch(r["variant"], first_epoch, N_GLOBAL_COMPONENTS)
    init_bases.append(basis)
    print(f"  {r['label']:20s}  epoch {first_epoch}")

init_angle_matrix = pairwise_principal_angles(init_bases)

Extracting epoch-0 W_in subspaces...
  s485/ds598            epoch 0
  s999/ds598            epoch 0
  s485/ds42             epoch 0
  s485/ds999            epoch 0
  s999/ds42             epoch 0
  s999/ds999            epoch 0


In [13]:
seed_labels_full = [
    f"{r['label']} ({'H' if r['health']=='healthy' else 'D'})"
    for r in records
]

for mat, title in [
    (init_angle_matrix,   f"p={PRIME} — W_in subspace alignment at epoch 0"),
    (global_angle_matrix, f"p={PRIME} — W_in subspace alignment at final epoch"),
]:
    fig = go.Figure(go.Heatmap(
        z=mat,
        x=seed_labels_full, y=seed_labels_full,
        colorscale="RdBu_r",
        zmid=45, zmin=0, zmax=90,
        text=np.round(mat, 1),
        texttemplate="%{text}°",
        colorbar=dict(title="Mean principal<br>angle (°)"),
    ))
    fig.update_layout(
        title=title,
        width=680, height=620,
        xaxis=dict(tickangle=-45),
        margin=dict(l=130, b=130),
    )
    fig.show()

# Numerical summary: same-seed pairs vs. different-seed pairs
same_seed_pairs = [
    (i, j) for i in range(n) for j in range(i + 1, n)
    if records[i]["variant"].params["seed"] == records[j]["variant"].params["seed"]
]
diff_seed_pairs = [
    (i, j) for i in range(n) for j in range(i + 1, n)
    if records[i]["variant"].params["seed"] != records[j]["variant"].params["seed"]
]

def _pair_mean(pairs, mat):
    return float(np.mean([mat[i, j] for i, j in pairs])) if pairs else float("nan")

print(f"{'':25s}  {'epoch 0':>10}  {'final':>10}")
print(f"{'Same-seed pairs':25s}  {_pair_mean(same_seed_pairs, init_angle_matrix):>9.1f}°  {_pair_mean(same_seed_pairs, global_angle_matrix):>9.1f}°")
print(f"{'Diff-seed pairs':25s}  {_pair_mean(diff_seed_pairs, init_angle_matrix):>9.1f}°  {_pair_mean(diff_seed_pairs, global_angle_matrix):>9.1f}°")

                              epoch 0       final
Same-seed pairs                  2.0°       73.8°
Diff-seed pairs                 79.5°       80.1°


## Saddle coverage — same surface or different patches?

Subspace alignment (principal angles) tells us whether two variants' freq-X neurons span the
same 3-D subspace.  But aligned subspaces can still have neurons in different *regions* of
that subspace — different patches of the same saddle, or redundant coverage of the same patch.

**Approach:** pool all freq-X neurons from the within-seed variants, fit PCA on the combined
set to get a shared saddle coordinate frame, then project each variant's neurons separately.

- **Overlapping clouds** → same patch, redundant coverage
- **Separated clusters** → different saddles (subspace alignment was coincidental)
- **Complementary regions** → different patches of the same saddle surface

In [14]:
_COVERAGE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

def load_freq_neurons(variant, freq: int) -> np.ndarray | None:
    """Final-epoch W_in rows for neurons assigned to freq. Returns (n, d_model) or None."""
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    epochs = loader.get_epochs("parameter_snapshot")
    snap = loader.load_epoch("parameter_snapshot", epochs[-1])
    W_in = extract_neuron_weight_matrix(snap)          # (d_model, d_mlp)

    ngpca = loader.load_cross_epoch("neuron_group_pca")
    group_idx   = ngpca["neuron_group_idx"]
    group_freqs = (ngpca["group_freqs"] + 1).astype(int)

    g_matches = np.where(group_freqs == freq)[0]
    if len(g_matches) == 0:
        return None
    mask = group_idx == g_matches[0]
    return W_in.T[mask]                                # (n_neurons, d_model)


def plot_saddle_coverage(seed_records: list, freq: int, seed: int):
    """Pool freq neurons across variants, project into shared PCA, overlay 3-D scatter."""
    neuron_blocks, labels = [], []
    for r in seed_records:
        neurons = load_freq_neurons(r["variant"], freq)
        if neurons is None:
            continue
        neuron_blocks.append(neurons)
        labels.append(r["label"])

    if len(neuron_blocks) < 2:
        print(f"  freq {freq}: fewer than 2 variants have this frequency — skipping")
        return

    pooled = np.vstack(neuron_blocks)          # (total_neurons, d_model)
    pca = PCA(n_components=3, random_state=0)
    all_coords = pca.fit_transform(pooled)     # shared saddle frame
    vr = pca.explained_variance_ratio_

    # Split back into per-variant blocks
    traces = []
    offset = 0
    for i, (neurons, label) in enumerate(zip(neuron_blocks, labels)):
        n = len(neurons)
        pts = all_coords[offset : offset + n]
        offset += n
        color = _COVERAGE_COLORS[i % len(_COVERAGE_COLORS)]
        traces.append(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=3, color=color, opacity=0.7),
            name=f"{label} (n={n})",
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=(
            f"p={PRIME} seed={seed} freq {freq} — pooled saddle coverage<br>"
            f"<sup>shared PCA: PC1={vr[0]:.1%} PC2={vr[1]:.1%} PC3={vr[2]:.1%}</sup>"
        ),
        scene=dict(
            xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3",
            aspectmode="cube",
        ),
        height=580, width=680,
        legend=dict(x=1.0, y=0.9),
    )
    fig.show()


for seed, seed_records in sorted(by_seed.items()):
    freq_sets = [set(r["group_subspaces"].keys()) for r in seed_records]
    anchor_freqs = sorted(set.intersection(*freq_sets))
    print(f"seed={seed} — anchor freqs: {anchor_freqs}")
    for freq in anchor_freqs:
        plot_saddle_coverage(seed_records, freq, seed)

seed=485 — anchor freqs: [36]


seed=999 — anchor freqs: []


## Saddle shape parameters across geometric realizations

The subspace alignment shows freq 36 neurons in different variants occupy near-orthogonal
weight-space directions — geometrically distinct structures with the same Fourier label.

Do the **saddle shape parameters** (a, b, c, det(H), R²) agree across these realizations?

- If yes: the saddle shape is determined by the frequency/task, independent of geometric path
- If no: the shape depends on the specific realization — multiple distinct saddle geometries exist for the same frequency

In [15]:
def load_final_shape_params(variant, freq: int) -> dict | None:
    """Final-epoch saddle shape parameters for a given frequency group."""
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))

    ngpca = loader.load_cross_epoch("neuron_group_pca")
    group_freqs = (ngpca["group_freqs"] + 1).astype(int)  # 1-indexed
    g_matches = np.where(group_freqs == freq)[0]
    if len(g_matches) == 0:
        return None
    g = int(g_matches[0])

    manifold = loader.load_cross_epoch("intragroup_manifold")
    r2 = float(manifold["r2_curvature"][-1, g])
    a  = float(manifold["a"][-1, g])
    b  = float(manifold["b"][-1, g])
    c  = float(manifold["c"][-1, g])
    det_h = 4 * a * b - c ** 2
    n_neurons = int((ngpca["neuron_group_idx"] == g).sum())

    return {"r2": r2, "a": a, "b": b, "c": c, "det_h": det_h, "n": n_neurons}


# Compare shape params for each anchor freq across within-seed variants
print(f"{'Variant':20s}  {'freq':>5}  {'n':>5}  {'R²':>6}  {'a':>7}  {'b':>7}  {'c':>7}  {'det(H)':>8}  shape")
print("-" * 82)

for seed, seed_records in sorted(by_seed.items()):
    freq_sets = [set(r["group_subspaces"].keys()) for r in seed_records]
    anchor_freqs = sorted(set.intersection(*freq_sets))
    for freq in anchor_freqs:
        for r in seed_records:
            p = load_final_shape_params(r["variant"], freq)
            if p is None:
                continue
            shape = "saddle" if p["det_h"] < 0 else "bowl"
            print(
                f"{r['label']:20s}  {freq:>5}  {p['n']:>5}  {p['r2']:>6.3f}"
                f"  {p['a']:>7.3f}  {p['b']:>7.3f}  {p['c']:>7.3f}"
                f"  {p['det_h']:>8.3f}  {shape}"
            )
        print()

Variant                freq      n      R²        a        b        c    det(H)  shape
----------------------------------------------------------------------------------
s485/ds598               36    210   0.665    0.354   -0.049   -2.232    -5.050  saddle
s485/ds42                36    113   0.808   -0.296    0.323    0.930    -1.248  saddle
s485/ds999               36    111   0.070    0.068    0.039    0.006     0.010  bowl



## Subspace alignment over training

Does the divergence between variants happen at initialization, at grokking, or gradually?

Two views:
1. **By epoch** — raw alignment trajectory for every variant pair
2. **Phase-normalized** — epoch divided by grokking epoch, so models are compared at
   equivalent developmental stages rather than equivalent wall-clock epochs

Pairs are colored by relationship: same-seed (should start at ~0° and diverge) vs.
different-seed (should start at ~80° and potentially change shape during training).

In [16]:
def get_grokking_epoch(variant) -> float | None:
    """Grokking epoch from variant_summary, or None if not available."""
    summary_path = variant.variant_dir / "variant_summary.json"
    if not summary_path.exists():
        return None
    summary = json.loads(summary_path.read_text())
    return summary.get("test_loss_threshold_first_epoch")


def compute_alignment_over_training(records: list, n_components: int) -> dict:
    """Pairwise subspace angles at every shared checkpoint.

    Returns:
        epochs:       sorted list of epoch numbers present in all variants
        pairs:        list of (i, j, label, same_seed) for each unique pair
        angle_traces: {(i,j): list of mean principal angles, one per epoch}
    """
    # Find epochs present in ALL variants
    epoch_sets = []
    for r in records:
        loader = ArtifactLoader(str(r["variant"].variant_dir / "artifacts"))
        epoch_sets.append(set(loader.get_epochs("parameter_snapshot")))
    shared_epochs = sorted(set.intersection(*epoch_sets))
    print(f"Shared checkpoints: {len(shared_epochs)} epochs")

    # Extract basis at every shared epoch for every variant
    print("Extracting subspace bases across training...")
    bases_by_epoch = {}  # {epoch: [basis_r0, basis_r1, ...]}
    for epoch in shared_epochs:
        bases_by_epoch[epoch] = []
        for r in records:
            basis = extract_win_subspace_at_epoch(r["variant"], epoch, n_components)
            bases_by_epoch[epoch].append(basis)
        print(f"  epoch {epoch:>6d} ✓", end="\r")
    print()

    # Build pair metadata
    n = len(records)
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            same_seed = (
                records[i]["variant"].params["seed"] ==
                records[j]["variant"].params["seed"]
            )
            label = f"{records[i]['label']} / {records[j]['label']}"
            pairs.append((i, j, label, same_seed))

    # Compute pairwise angles at each epoch
    angle_traces = {}
    for (i, j, label, same_seed) in pairs:
        angles = []
        for epoch in shared_epochs:
            A, B = bases_by_epoch[epoch][i], bases_by_epoch[epoch][j]
            val = float(np.degrees(subspace_angles(A.T, B.T).mean()))
            angles.append(val)
        angle_traces[(i, j)] = angles

    return {"epochs": shared_epochs, "pairs": pairs, "angle_traces": angle_traces}


alignment = compute_alignment_over_training(records, N_GLOBAL_COMPONENTS)

Shared checkpoints: 95 epochs
Extracting subspace bases across training...
  epoch  24999 ✓


In [17]:
epochs = alignment["epochs"]
pairs  = alignment["pairs"]
traces = alignment["angle_traces"]

_SAME_SEED_COLORS  = ["#1f77b4", "#aec7e8", "#6baed6"]   # blues
_DIFF_SEED_COLORS  = ["#d62728", "#e6550d", "#fd8d3c",
                      "#e377c2", "#c5b0d5", "#bcbd22",
                      "#17becf", "#98df8a", "#ffbb78"]

same_idx, diff_idx = 0, 0
fig = go.Figure()

for (i, j, label, same_seed) in pairs:
    if same_seed:
        color = _SAME_SEED_COLORS[same_idx % len(_SAME_SEED_COLORS)]
        same_idx += 1
        dash, width = "solid", 2
    else:
        color = _DIFF_SEED_COLORS[diff_idx % len(_DIFF_SEED_COLORS)]
        diff_idx += 1
        dash, width = "dot", 1

    fig.add_trace(go.Scatter(
        x=epochs, y=traces[(i, j)],
        mode="lines",
        name=label,
        line=dict(color=color, dash=dash, width=width),
    ))

fig.update_layout(
    title=f"p={PRIME} — W_in subspace alignment over training ({N_GLOBAL_COMPONENTS}-D)",
    xaxis_title="Epoch",
    yaxis_title="Mean principal angle (°)",
    yaxis_range=[0, 90],
    height=480, width=860,
    legend=dict(font=dict(size=10)),
)
fig.show()

In [18]:
# Phase-normalized view: x = epoch / grokking_epoch
# Variants that never grokked (grokking_epoch <= 0 or None) are excluded from this plot.
for r in records:
    r["grokking_epoch"] = get_grokking_epoch(r["variant"])
    status = str(r["grokking_epoch"]) if r["grokking_epoch"] and r["grokking_epoch"] > 0 else "never grokked"
    print(f"  {r['label']:20s}  grokking epoch: {status}")

fig2 = go.Figure()
same_idx, diff_idx = 0, 0
skipped = []

for (i, j, label, same_seed) in pairs:
    gi = records[i]["grokking_epoch"]
    gj = records[j]["grokking_epoch"]
    if not gi or not gj or gi <= 0 or gj <= 0:
        skipped.append(label)
        continue

    x_norm = [(e / gi + e / gj) / 2 for e in epochs]

    if same_seed:
        color = _SAME_SEED_COLORS[same_idx % len(_SAME_SEED_COLORS)]
        same_idx += 1
        dash, width = "solid", 2
    else:
        color = _DIFF_SEED_COLORS[diff_idx % len(_DIFF_SEED_COLORS)]
        diff_idx += 1
        dash, width = "dot", 1

    fig2.add_trace(go.Scatter(
        x=x_norm, y=traces[(i, j)],
        mode="lines",
        name=label,
        line=dict(color=color, dash=dash, width=width),
    ))

if skipped:
    print(f"\nSkipped (never grokked): {skipped}")

fig2.add_vline(x=1.0, line=dict(color="black", dash="dash", width=1),
               annotation_text="grokking", annotation_position="top right")
fig2.update_layout(
    title=f"p={PRIME} — subspace alignment, phase-normalized (epoch / grokking_epoch)",
    xaxis_title="Phase (epoch / grokking_epoch)",
    yaxis_title="Mean principal angle (°)",
    yaxis_range=[0, 90],
    xaxis_range=[0, None],
    height=480, width=860,
    legend=dict(font=dict(size=10)),
)
fig2.show()

  s485/ds598            grokking epoch: 10113
  s999/ds598            grokking epoch: 12448
  s485/ds42             grokking epoch: 21396
  s485/ds999            grokking epoch: never grokked
  s999/ds42             grokking epoch: 26946
  s999/ds999            grokking epoch: never grokked

Skipped (never grokked): ['s485/ds598 / s485/ds999', 's485/ds598 / s999/ds999', 's999/ds598 / s485/ds999', 's999/ds598 / s999/ds999', 's485/ds42 / s485/ds999', 's485/ds42 / s999/ds999', 's485/ds999 / s999/ds42', 's485/ds999 / s999/ds999', 's999/ds42 / s999/ds999']


## Pinned-checkpoint comparison

Wall-clock epoch and grokking-normalized phase both compare models at the same number,
but different models reach qualitatively equivalent moments (L2 trough, R² peak, wings fully
risen) at different epochs.  This cell lets you pin each variant to a specific checkpoint —
the epoch at which you believe it is at a comparable developmental stage.

Set `PINNED_EPOCHS` to the epoch you want for each variant label.  Variants not listed
fall back to their final epoch.  Run the alignment training cells first so `records` is
populated with `label` keys.

In [ ]:
# ── Edit this dict to pin each variant to a specific epoch ───────────────────
# Keys match the "label" field (e.g. "s485/ds598").
# Unlisted variants use their final available checkpoint.
# Suggested moments to compare:
#   - L2 spread trough  (flat-disk maximum, just before wings rising)
#   - R² peak           (saddle at maximum extent)
#   - post-grokking     (some fixed multiple of grokking epoch, e.g. 2×)
PINNED_EPOCHS: dict[str, int] = {
    "s485/ds598": 16000,
    #"s999/ds598": 16000,
    "s485/ds42":  19000,
    # ...
}
# ─────────────────────────────────────────────────────────────────────────────

def resolve_pinned_epoch(variant, label: str, pinned: dict) -> int:
    """Return the pinned epoch for a variant, or its final checkpoint."""
    if label in pinned:
        return pinned[label]
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    return loader.get_epochs("parameter_snapshot")[-1]


print("Pinned checkpoints:")
pinned_bases = []
pinned_epoch_used = []
for r in records:
    epoch = resolve_pinned_epoch(r["variant"], r["label"], PINNED_EPOCHS)
    basis = extract_win_subspace_at_epoch(r["variant"], epoch, N_GLOBAL_COMPONENTS)
    pinned_bases.append(basis)
    pinned_epoch_used.append(epoch)
    pin_note = " (pinned)" if r["label"] in PINNED_EPOCHS else " (final)"
    print(f"  {r['label']:20s}  epoch {epoch}{pin_note}")

pinned_angle_matrix = pairwise_principal_angles(pinned_bases)

fig3 = go.Figure(go.Heatmap(
    z=pinned_angle_matrix,
    x=[f"{r['label']} @{e}" for r, e in zip(records, pinned_epoch_used)],
    y=[f"{r['label']} @{e}" for r, e in zip(records, pinned_epoch_used)],
    colorscale="RdBu_r",
    zmid=45, zmin=0, zmax=90,
    text=np.round(pinned_angle_matrix, 1),
    texttemplate="%{text}°",
    colorbar=dict(title="Mean principal<br>angle (°)"),
))
fig3.update_layout(
    title=f"p={PRIME} — pinned-checkpoint subspace alignment ({N_GLOBAL_COMPONENTS}-D)",
    width=720, height=650,
    xaxis=dict(tickangle=-45),
    margin=dict(l=140, b=140),
)
fig3.show()

Pinned checkpoints:
  s485/ds598            epoch 24999 (final)
  s999/ds598            epoch 24999 (final)
  s485/ds42             epoch 24999 (final)
  s485/ds999            epoch 24999 (final)
  s999/ds42             epoch 34999 (final)
  s999/ds999            epoch 24999 (final)
